# 📊 Complete Guide to Linear Regression

## 🎯 Learning Objectives
By the end of this notebook, you will understand:
- The mathematical foundations of linear regression
- Different approaches to solve linear regression (Normal Equation vs Gradient Descent)
- How to implement and evaluate linear regression models
- Advanced topics: regularization, feature engineering, and model diagnostics
- Real-world applications and best practices

---

## 📚 Table of Contents
1. [Mathematical Foundation](#mathematical-foundation)
2. [Implementation Approaches](#implementation-approaches)
3. [Data Preparation & Exploration](#data-preparation)
4. [Model Implementation](#model-implementation)
5. [Advanced Topics](#advanced-topics)
6. [Model Evaluation & Diagnostics](#model-evaluation)
7. [Real-World Applications](#real-world-applications)

---

## 🧮 Mathematical Foundation

### What is Linear Regression?

Linear regression is a statistical method that models the relationship between a **dependent variable** (target) and one or more **independent variables** (features) using a linear equation.

### Simple Linear Regression (One Feature)
For one predictor variable, the model is:

$$h_\theta(x) = \theta_0 + \theta_1 x$$

Where:
- $h_\theta(x)$ = predicted value
- $\theta_0$ = intercept (bias term)
- $\theta_1$ = slope (weight)
- $x$ = input feature

### Multiple Linear Regression (Multiple Features)
For multiple predictors:

$$h_\theta(x) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \ldots + \theta_n x_n$$

Or in matrix form:
$$h_\theta(x) = X\theta$$

Where:
- $X$ = feature matrix (m × n+1, including bias column)
- $\theta$ = parameter vector (n+1 × 1)
- $m$ = number of training examples
- $n$ = number of features

### Cost Function (Mean Squared Error)
We minimize the cost function to find the best parameters:

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2$$

### Key Assumptions
1. **Linearity**: Relationship between X and y is linear
2. **Independence**: Observations are independent
3. **Homoscedasticity**: Constant variance of residuals
4. **Normality**: Residuals are normally distributed (for inference)

---

## 🔧 Implementation Approaches

There are two main approaches to solve linear regression:

### 1. Normal Equation (Closed-form Solution)
The analytical solution that directly computes optimal parameters:

$$\theta = (X^T X)^{-1} X^T y$$

**Advantages:**
- ✅ No need to choose learning rate
- ✅ No iterations needed
- ✅ Exact solution

**Disadvantages:**
- ❌ Slow when n > 10,000 features (O(n³) complexity)
- ❌ Requires matrix inversion
- ❌ X^T X must be invertible

### 2. Gradient Descent (Iterative Solution)
Iteratively updates parameters by moving in the direction of steepest descent:

$$\theta_j := \theta_j - \alpha \frac{\partial}{\partial \theta_j} J(\theta)$$

Where the gradient is:
$$\frac{\partial}{\partial \theta_j} J(\theta) = \frac{1}{m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)}) x_j^{(i)}$$

**Advantages:**
- ✅ Works well even when n is large
- ✅ Can be used for other algorithms
- ✅ Memory efficient

**Disadvantages:**
- ❌ Need to choose learning rate α
- ❌ Needs many iterations
- ❌ May not converge if α is too large

### When to Use Which?
- **Normal Equation**: When n ≤ 10,000 and X^T X is invertible
- **Gradient Descent**: When n > 10,000 or for learning purposes

---

## 📊 Import Libraries and Setup

In [ ]:
# Essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.datasets import make_regression
import warnings
import time
from scipy import stats
from scipy.stats import jarque_bera, shapiro

warnings.filterwarnings('ignore')

# Configuration for better visualizations
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
sns.set_palette("husl")

print("🚀 Linear Regression Comprehensive Analysis")
print("=" * 60)
print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
print("=" * 60)

## 📊 Data Exploration and Visualization

In [ ]:
# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
try:
    data = pd.read_csv(url, delimiter=";")
    print("✅ Dataset loaded successfully from UCI repository")
except:
    print("❌ Could not load from URL, creating synthetic dataset...")
    # Create synthetic dataset as fallback
    X_synthetic, y_synthetic = make_regression(n_samples=1000, n_features=11, 
                                             noise=0.1, random_state=42)
    feature_names = ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
                    'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 
                    'density', 'pH', 'sulphates', 'alcohol']
    data = pd.DataFrame(X_synthetic, columns=feature_names)
    data['quality'] = y_synthetic

print(f"\n📋 Dataset Overview:")
print(f"   • Shape: {data.shape}")
print(f"   • Features: {data.shape[1]-1}")
print(f"   • Samples: {data.shape[0]}")

# Check for missing values
missing_info = data.isnull().sum()
if missing_info.sum() > 0:
    print(f"\n⚠️  Missing values detected:")
    print(missing_info[missing_info > 0])
else:
    print("\n✅ No missing values detected")

# Feature names and target
feature_names = list(data.columns[:-1])
target_name = data.columns[-1]
print(f"\n🎯 Target variable: {target_name}")
print(f"📊 Features: {', '.join(feature_names[:3])}... ({len(feature_names)} total)")

# Display first few rows
print(f"\n📋 First 3 rows:")
display(data.head(3))

# Prepare data for modeling
X = data.iloc[:, :-1].values  # Features
y = data.iloc[:, -1].values   # Target

print(f"\n🔢 Data prepared:")
print(f"   • X shape: {X.shape}")
print(f"   • y shape: {y.shape}")

In [ ]:
# Split the data first (IMPORTANT: Always split before preprocessing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                  random_state=42, stratify=None)

print(f"📊 Data Split:")
print(f"   • Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   • Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Feature scaling (fit on training data only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n📏 Feature Scaling Applied:")
print(f"   • Method: StandardScaler (mean=0, std=1)")
print(f"   • Training mean: {X_train_scaled.mean():.2e}")
print(f"   • Training std: {X_train_scaled.std():.2f}")

# Add bias term for matrix operations
X_train_bias = np.hstack((np.ones((X_train_scaled.shape[0], 1)), X_train_scaled))
X_test_bias = np.hstack((np.ones((X_test_scaled.shape[0], 1)), X_test_scaled))

print(f"\n🔢 Final shapes (with bias term):")
print(f"   • X_train_bias: {X_train_bias.shape}")
print(f"   • X_test_bias: {X_test_bias.shape}")
print(f"   • y_train: {y_train.shape}")
print(f"   • y_test: {y_test.shape}")

# Store feature names for later use
if 'feature_names' in locals():
    feature_names_with_bias = ['intercept'] + feature_names
else:
    feature_names_with_bias = ['intercept'] + [f'feature_{i}' for i in range(X.shape[1])]

print(f"\n📝 Features (with bias): {len(feature_names_with_bias)} total")

In [ ]:
# =====================================
# APPROACH 1: NORMAL EQUATION
# =====================================

def normal_equation(X, y):
    """
    Compute theta using the normal equation: θ = (X^T X)^-1 X^T y
    """
    start_time = time.time()
    
    # Check if X^T X is invertible
    XtX = X.T @ X
    try:
        theta = np.linalg.inv(XtX) @ X.T @ y
        computation_time = time.time() - start_time
        return theta, computation_time, True
    except np.linalg.LinAlgError:
        print("⚠️  Matrix X^T X is not invertible! Using pseudo-inverse...")
        theta = np.linalg.pinv(XtX) @ X.T @ y
        computation_time = time.time() - start_time
        return theta, computation_time, False

# Apply Normal Equation
theta_normal, time_normal, invertible = normal_equation(X_train_bias, y_train)

print(f"✅ Normal Equation completed!")
print(f"   • Computation time: {time_normal:.4f} seconds")
print(f"   • Matrix invertible: {'Yes' if invertible else 'No'}")
print(f"   • Parameters shape: {theta_normal.shape}")

# =====================================
# APPROACH 2: GRADIENT DESCENT
# =====================================

class GradientDescentRegressor:
    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6):
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.tolerance = tolerance
        self.cost_history = []
        self.theta_history = []
        
    def compute_cost(self, X, y, theta):
        """Compute the cost function (MSE)"""
        m = len(y)
        predictions = X @ theta
        cost = (1 / (2 * m)) * np.sum((predictions - y) ** 2)
        return cost
    
    def fit(self, X, y):
        """Fit the model using gradient descent"""
        m, n = X.shape
        self.theta = np.zeros(n)
        
        start_time = time.time()
        
        for i in range(self.max_iterations):
            # Forward pass
            predictions = X @ self.theta
            errors = predictions - y
            
            # Compute cost
            cost = self.compute_cost(X, y, self.theta)
            self.cost_history.append(cost)
            self.theta_history.append(self.theta.copy())
            
            # Compute gradients
            gradients = (1 / m) * (X.T @ errors)
            
            # Update parameters
            new_theta = self.theta - self.learning_rate * gradients
            
            # Check for convergence
            if np.linalg.norm(new_theta - self.theta) < self.tolerance:
                print(f"   • Converged at iteration {i+1}")
                break
                
            self.theta = new_theta
        
        self.computation_time = time.time() - start_time
        return self
    
    def predict(self, X):
        return X @ self.theta

# Train using Gradient Descent
gd_regressor = GradientDescentRegressor(learning_rate=0.01, max_iterations=2000)
gd_regressor.fit(X_train_bias, y_train)

print(f"✅ Gradient Descent completed!")
print(f"   • Computation time: {gd_regressor.computation_time:.4f} seconds")
print(f"   • Total iterations: {len(gd_regressor.cost_history)}")
print(f"   • Final cost: {gd_regressor.cost_history[-1]:.6f}")

# =====================================
# APPROACH 3: SCIKIT-LEARN
# =====================================

start_time = time.time()
sklearn_regressor = LinearRegression()
sklearn_regressor.fit(X_train_scaled, y_train)
time_sklearn = time.time() - start_time

# Extract parameters (need to add intercept manually for comparison)
theta_sklearn = np.concatenate([[sklearn_regressor.intercept_], sklearn_regressor.coef_])

print(f"✅ Scikit-learn completed!")
print(f"   • Computation time: {time_sklearn:.4f} seconds")
print(f"   • R² score (training): {sklearn_regressor.score(X_train_scaled, y_train):.4f}")

# =====================================
# COMPARISON OF APPROACHES
# =====================================

approaches = {
    'Normal Equation': {'theta': theta_normal, 'time': time_normal},
    'Gradient Descent': {'theta': gd_regressor.theta, 'time': gd_regressor.computation_time},
    'Scikit-learn': {'theta': theta_sklearn, 'time': time_sklearn}
}

print(f"\n🔍 COMPARISON OF APPROACHES")
print("=" * 50)
print(f"{'Method':<20} {'Time (s)':<12} {'Intercept':<12} {'Feature 1':<12}")
print("-" * 60)
for name, info in approaches.items():
    theta = info['theta']
    time_taken = info['time']
    print(f"{name:<20} {time_taken:<12.4f} {theta[0]:<12.3f} {theta[1]:<12.3f}")

# Check parameter similarity
print(f"\n🎯 Parameter Similarity Check:")
diff_norm_gd = np.linalg.norm(theta_normal - gd_regressor.theta)
diff_norm_sklearn = np.linalg.norm(theta_normal - theta_sklearn)
print(f"   • Normal vs Gradient Descent: {diff_norm_gd:.6f}")
print(f"   • Normal vs Scikit-learn: {diff_norm_sklearn:.6f}")
print(f"   • {'✅ All methods converged to similar solution!' if max(diff_norm_gd, diff_norm_sklearn) < 0.01 else '⚠️  Methods show different results!'}")

In [ ]:
# 🔍 STEP 5: COMPREHENSIVE MODEL EVALUATION
print("\n" + "="*60)
print("🔍 STEP 5: COMPREHENSIVE MODEL EVALUATION")
print("="*60)

# Make predictions using all approaches
predictions_normal_train = X_train_bias @ theta_normal
predictions_normal_test = X_test_bias @ theta_normal

predictions_gd_train = gd_regressor.predict(X_train_bias)
predictions_gd_test = gd_regressor.predict(X_test_bias)

predictions_sklearn_train = sklearn_regressor.predict(X_train_scaled)
predictions_sklearn_test = sklearn_regressor.predict(X_test_scaled)

# Calculate comprehensive metrics
def calculate_metrics(y_true, y_pred):
    """Calculate comprehensive regression metrics"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # Additional metrics
    n = len(y_true)
    p = 1  # Number of predictors (simplified)
    adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    
    # Mean Absolute Percentage Error (MAPE)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true != 0, y_true, 1))) * 100
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'Adjusted R²': adjusted_r2,
        'MAPE': mape
    }

# Evaluate all models
models_results = {
    'Normal Equation': {
        'train': calculate_metrics(y_train, predictions_normal_train),
        'test': calculate_metrics(y_test, predictions_normal_test)
    },
    'Gradient Descent': {
        'train': calculate_metrics(y_train, predictions_gd_train),
        'test': calculate_metrics(y_test, predictions_gd_test)
    },
    'Scikit-learn': {
        'train': calculate_metrics(y_train, predictions_sklearn_train),
        'test': calculate_metrics(y_test, predictions_sklearn_test)
    }
}

# Display results in a nice table format
print("\n📊 PERFORMANCE METRICS COMPARISON")
print("=" * 80)
print(f"{'Model':<15} {'Dataset':<8} {'MSE':<8} {'RMSE':<8} {'MAE':<8} {'R²':<8} {'MAPE':<8}")
print("-" * 80)

for model_name, results in models_results.items():
    for dataset_type, metrics in results.items():
        print(f"{model_name:<15} {dataset_type:<8} {metrics['MSE']:<8.4f} {metrics['RMSE']:<8.4f} "
              f"{metrics['MAE']:<8.4f} {metrics['R²']:<8.4f} {metrics['MAPE']:<8.2f}%")

# Overfitting Analysis
print(f"\n🎯 OVERFITTING ANALYSIS")
print("-" * 30)
for model_name, results in models_results.items():
    train_r2 = results['train']['R²']
    test_r2 = results['test']['R²']
    r2_diff = abs(train_r2 - test_r2)
    
    if r2_diff < 0.05:
        status = "✅ Good generalization"
    elif r2_diff < 0.1:
        status = "⚠️  Slight overfitting"
    else:
        status = "❌ Overfitting detected"
    
    print(f"{model_name:<15}: Train R²={train_r2:.4f}, Test R²={test_r2:.4f}, "
          f"Diff={r2_diff:.4f} - {status}")

print(f"\n🏆 BEST MODEL SELECTION")
print("-" * 25)
# Select best model based on test R² score
best_model = max(models_results.keys(), 
                key=lambda x: models_results[x]['test']['R²'])
best_r2 = models_results[best_model]['test']['R²']
print(f"Best performing model: {best_model}")
print(f"Test R² Score: {best_r2:.4f}")

# For detailed analysis, we'll use the Normal Equation results (most straightforward)
train_pred_final = predictions_normal_train
test_pred_final = predictions_normal_test

In [ ]:
# Import required libraries for regularization
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures

# 🚀 STEP 7: ADVANCED TOPICS AND REGULARIZATION
print("\n" + "="*60)
print("🚀 STEP 7: ADVANCED TOPICS AND REGULARIZATION")
print("="*60)

# =====================================
# REGULARIZED LINEAR REGRESSION
# =====================================
print("\n🛡️  REGULARIZATION TECHNIQUES")
print("-" * 40)

# Ridge Regression (L2 Regularization)
ridge_alphas = [0.1, 1.0, 10.0, 100.0]
ridge_results = {}

print("🔵 Ridge Regression (L2 Regularization)")
for alpha in ridge_alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    
    train_score = ridge.score(X_train_scaled, y_train)
    test_score = ridge.score(X_test_scaled, y_test)
    
    ridge_results[alpha] = {
        'train_r2': train_score,
        'test_r2': test_score,
        'coef_': ridge.coef_,
        'intercept_': ridge.intercept_
    }
    
    print(f"   α={alpha:6.1f}: Train R²={train_score:.4f}, Test R²={test_score:.4f}")

# Lasso Regression (L1 Regularization)
lasso_alphas = [0.01, 0.1, 1.0, 10.0]
lasso_results = {}

print(f"\n🔴 Lasso Regression (L1 Regularization)")
for alpha in lasso_alphas:
    lasso = Lasso(alpha=alpha, max_iter=2000)
    lasso.fit(X_train_scaled, y_train)
    
    train_score = lasso.score(X_train_scaled, y_train)
    test_score = lasso.score(X_test_scaled, y_test)
    
    # Count non-zero coefficients (feature selection)
    non_zero_coefs = np.sum(np.abs(lasso.coef_) > 1e-5)
    
    lasso_results[alpha] = {
        'train_r2': train_score,
        'test_r2': test_score,
        'coef_': lasso.coef_,
        'intercept_': lasso.intercept_,
        'n_features': non_zero_coefs
    }
    
    print(f"   α={alpha:6.2f}: Train R²={train_score:.4f}, Test R²={test_score:.4f}, "
          f"Active features: {non_zero_coefs}")

# =====================================
# CROSS-VALIDATION
# =====================================
print(f"\n🎯 CROSS-VALIDATION ANALYSIS")
print("-" * 30)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(LinearRegression(), X_train_scaled, y_train, cv=5, scoring='r2')

print(f"5-Fold Cross-Validation Results:")
print(f"   • CV Scores: {[f'{score:.4f}' for score in cv_scores]}")
print(f"   • Mean CV Score: {cv_scores.mean():.4f} (±{cv_scores.std()*2:.4f})")
print(f"   • Best Fold: {cv_scores.max():.4f}")
print(f"   • Worst Fold: {cv_scores.min():.4f}")

# =====================================
# POLYNOMIAL FEATURES
# =====================================
print(f"\n🔢 POLYNOMIAL FEATURE ENGINEERING")
print("-" * 35)

# Try polynomial features (degree 2)
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly_features.fit_transform(X_train_scaled)
X_test_poly = poly_features.transform(X_test_scaled)

print(f"Original features: {X_train_scaled.shape[1]}")
print(f"Polynomial features (degree 2): {X_train_poly.shape[1]}")

# Train polynomial regression
poly_regressor = LinearRegression()
poly_regressor.fit(X_train_poly, y_train)

poly_train_score = poly_regressor.score(X_train_poly, y_train)
poly_test_score = poly_regressor.score(X_test_poly, y_test)

print(f"Polynomial Regression Results:")
print(f"   • Training R²: {poly_train_score:.4f}")
print(f"   • Test R²: {poly_test_score:.4f}")

# Check for overfitting
poly_overfit = abs(poly_train_score - poly_test_score)
if poly_overfit > 0.1:
    print(f"   ⚠️  Overfitting detected (difference: {poly_overfit:.4f})")
else:
    print(f"   ✅ Good generalization (difference: {poly_overfit:.4f})")

# Regularized polynomial regression
poly_ridge = Ridge(alpha=1.0)
poly_ridge.fit(X_train_poly, y_train)

poly_ridge_train = poly_ridge.score(X_train_poly, y_train)
poly_ridge_test = poly_ridge.score(X_test_poly, y_test)

print(f"Regularized Polynomial Regression (Ridge α=1.0):")
print(f"   • Training R²: {poly_ridge_train:.4f}")
print(f"   • Test R²: {poly_ridge_test:.4f}")
print(f"   • Overfitting reduced: {abs(poly_ridge_train - poly_ridge_test):.4f}")

In [ ]:
# 📊 STEP 6: MODEL DIAGNOSTICS AND VISUALIZATION
print("\n" + "="*60)
print("📊 STEP 6: MODEL DIAGNOSTICS AND VISUALIZATION")
print("="*60)

# Calculate residuals
train_residuals = y_train - train_pred_final
test_residuals = y_test - test_pred_final

fig, axes = plt.subplots(3, 2, figsize=(15, 18))

# 1. Actual vs Predicted (Training)
axes[0, 0].scatter(y_train, train_pred_final, alpha=0.6, color='blue', s=30)
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
                'r--', lw=2, label='Perfect Prediction')
axes[0, 0].set_title('🎯 Training: Actual vs Predicted', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Actual Values')
axes[0, 0].set_ylabel('Predicted Values')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Add R² annotation
r2_train = models_results['Normal Equation']['train']['R²']
axes[0, 0].text(0.05, 0.95, f'R² = {r2_train:.4f}', transform=axes[0, 0].transAxes, 
                fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat'))

# 2. Actual vs Predicted (Test)
axes[0, 1].scatter(y_test, test_pred_final, alpha=0.6, color='green', s=30)
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
                'r--', lw=2, label='Perfect Prediction')
axes[0, 1].set_title('🎯 Test: Actual vs Predicted', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Actual Values')
axes[0, 1].set_ylabel('Predicted Values')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Add R² annotation
r2_test = models_results['Normal Equation']['test']['R²']
axes[0, 1].text(0.05, 0.95, f'R² = {r2_test:.4f}', transform=axes[0, 1].transAxes, 
                fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgreen'))

# 3. Residuals vs Predicted (Training)
axes[1, 0].scatter(train_pred_final, train_residuals, alpha=0.6, color='blue', s=30)
axes[1, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_title('📏 Training Residuals vs Predicted', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Predicted Values')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].grid(True, alpha=0.3)

# 4. Residuals vs Predicted (Test)
axes[1, 1].scatter(test_pred_final, test_residuals, alpha=0.6, color='green', s=30)
axes[1, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_title('📏 Test Residuals vs Predicted', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Predicted Values')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].grid(True, alpha=0.3)

# 5. Residuals Distribution (Training)
axes[2, 0].hist(train_residuals, bins=30, alpha=0.7, color='blue', edgecolor='black', density=True)
axes[2, 0].set_title('📊 Training Residuals Distribution', fontsize=14, fontweight='bold')
axes[2, 0].set_xlabel('Residuals')
axes[2, 0].set_ylabel('Density')

# Overlay normal distribution for comparison
mu, sigma = train_residuals.mean(), train_residuals.std()
x = np.linspace(train_residuals.min(), train_residuals.max(), 100)
axes[2, 0].plot(x, stats.norm.pdf(x, mu, sigma), 'r--', linewidth=2, label='Normal Distribution')
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

# 6. Q-Q Plot for Normality Check
axes[2, 1].set_title('📈 Q-Q Plot: Residuals Normality', fontsize=14, fontweight='bold')
stats.probplot(train_residuals, dist="norm", plot=axes[2, 1])
axes[2, 1].get_lines()[0].set_markerfacecolor('blue')
axes[2, 1].get_lines()[0].set_markeredgecolor('blue')
axes[2, 1].get_lines()[0].set_alpha(0.6)
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical Tests for Residuals
print(f"\n🔬 RESIDUAL ANALYSIS")
print("=" * 30)

# Normality tests
try:
    shapiro_stat, shapiro_p = shapiro(train_residuals)
    jb_stat, jb_p = jarque_bera(train_residuals)
    
    print(f"📊 Normality Tests (Training Residuals):")
    print(f"   • Shapiro-Wilk test: statistic={shapiro_stat:.4f}, p-value={shapiro_p:.4f}")
    print(f"   • Jarque-Bera test: statistic={jb_stat:.4f}, p-value={jb_p:.4f}")
    
    if shapiro_p > 0.05:
        print(f"   ✅ Residuals appear normally distributed (Shapiro-Wilk p > 0.05)")
    else:
        print(f"   ⚠️  Residuals may not be normally distributed (Shapiro-Wilk p ≤ 0.05)")
        
except Exception as e:
    print(f"   ⚠️  Could not perform normality tests: {e}")

# Residual statistics
print(f"\n📈 Residual Statistics:")
print(f"   • Mean: {train_residuals.mean():.6f} (should be ~0)")
print(f"   • Std: {train_residuals.std():.4f}")
print(f"   • Min: {train_residuals.min():.4f}")
print(f"   • Max: {train_residuals.max():.4f}")
print(f"   • Skewness: {stats.skew(train_residuals):.4f} (should be ~0)")
print(f"   • Kurtosis: {stats.kurtosis(train_residuals):.4f} (should be ~0)")

# Homoscedasticity check (constant variance)
print(f"\n🔍 Homoscedasticity Check:")
# Split residuals into two groups based on predicted values
median_pred = np.median(train_pred_final)
low_pred_residuals = train_residuals[train_pred_final <= median_pred]
high_pred_residuals = train_residuals[train_pred_final > median_pred]

var_ratio = np.var(high_pred_residuals) / np.var(low_pred_residuals)
print(f"   • Variance ratio (high/low predictions): {var_ratio:.4f}")
if 0.5 <= var_ratio <= 2.0:
    print(f"   ✅ Residuals show relatively constant variance")
else:
    print(f"   ⚠️  Potential heteroscedasticity detected")

## 🎯 Key Takeaways and Practical Insights

### ✅ **What We Accomplished:**

1. **Complete Mathematical Understanding**: 
   - Derived the mathematical foundations of linear regression
   - Implemented both analytical (Normal Equation) and iterative (Gradient Descent) solutions
   - Compared computational efficiency and numerical stability

2. **Comprehensive Model Evaluation**:
   - Multiple performance metrics (MSE, RMSE, MAE, R², MAPE)
   - Rigorous diagnostic testing (residual analysis, normality tests, homoscedasticity)
   - Cross-validation for robust performance assessment

3. **Advanced Techniques**:
   - Ridge regression (L2) for handling multicollinearity
   - Lasso regression (L1) for feature selection
   - Polynomial features for capturing non-linear relationships
   - Regularization to prevent overfitting

4. **Production-Ready Code**:
   - Proper data splitting and preprocessing
   - Scalable implementations that work with large datasets
   - Comprehensive error handling and validation

---

### 📊 **Model Performance Insights:**

**When Linear Regression Works Well:**
- Linear relationship between features and target
- Low noise in the data
- Sufficient training data relative to features
- Features are not highly correlated (multicollinearity)

**Warning Signs to Watch For:**
- R² difference > 0.1 between training and test (overfitting)
- Non-normal residuals (model assumptions violated)
- Heteroscedasticity (non-constant error variance)
- High condition number of X^T X (numerical instability)

---

### 🔍 **When to Use Each Approach:**

| Approach | Best For | Avoid When |
|----------|----------|------------|
| **Normal Equation** | n < 10,000 features, exact solution needed | Large n, singular matrices |
| **Gradient Descent** | Large datasets, learning process insight | Need exact solution quickly |
| **Ridge Regression** | Multicollinearity, many features | Sparse feature selection needed |
| **Lasso Regression** | Feature selection, sparse models | All features are important |
| **Polynomial Features** | Non-linear relationships | Limited data, risk of overfitting |

---

**💭 Remember: Linear regression is often the starting point, not the ending point. Master it well, and you'll have a solid foundation for more advanced techniques!**